# Week 1 — Prompt Experiment Notebook
**Candidate:** Tejas | **Track:** AI Engineer

**Goal:** Run the same request using three prompt styles and observe what changes in the LLM response.

---

## Setup
- Same request for all three runs: `"Read file python_info.txt"`
- CLI command used: `python -m app.main --mode once --prompt-style <style> --request "Read file python_info.txt"`
- Project root is the working directory.


In [ ]:
# Install dependencies if not already installed
# !pip install groq python-dotenv pydantic pydantic-settings httpx

import os
import sys
import json

# Add project root to path so we can import from 'app'
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from app.automation_agent import AutomationAgent

print("Setup complete — agent ready")

---
## Experiment 1 — Zero-Shot Prompting

**What is zero-shot?**  
We give the model the system prompt + the user request, with NO examples at all.  
The model has to figure out the right tool and JSON format on its own from the instructions only.

In [ ]:
agent = AutomationAgent.create()

result_zero_shot = agent.run(
    user_request="Read file python_info.txt",
    mode="once",
    prompt_style="zero-shot",
)

print("=== ZERO-SHOT RESULT ===")
print(agent.result_as_json(result_zero_shot))

### Zero-Shot Observations

*(Fill in after running the cell above)*

- **Tool selected:** `read_file` ✅
- **JSON shape:** Correct / had extra text before JSON
- **What I noticed:**
  - Model had only the system prompt + user line. No example JSON in the user message.
  - It picked `read_file` correctly. The JSON contained `path: python_info.txt`.
  - One time the reply had a bit of extra text before the `{`, but `_parse_json()` found the `{}` anyway.

**My takeaway:**  
Fast and simple. For a clear request like "read this file" zero-shot was enough.  
If the model returns messy JSON, zero-shot has no example to copy from so I rely more on the parser.

---
## Experiment 2 — Few-Shot Prompting

**What is few-shot?**  
We add 1 or more example request→answer pairs inside the user message **before** the real request.  
The model can copy the JSON format from the example instead of guessing.

How it looks in the prompt:
```
Example:
User: "Read file notes/today.txt"
Assistant: {"tool_name":"read_file","arguments":{"path":"notes/today.txt"},"reason":""}

User: Read file python_info.txt
```

In [ ]:
agent = AutomationAgent.create()

result_few_shot = agent.run(
    user_request="Read file python_info.txt",
    mode="once",
    prompt_style="few-shot",
)

print("=== FEW-SHOT RESULT ===")
print(agent.result_as_json(result_few_shot))

### Few-Shot Observations

- **Tool selected:** `read_file` ✅  
- **JSON shape:** More stable / same as before
- **What I noticed:**
  - Prompt included one example (read file → JSON). The model had a pattern to copy.
  - Same tool `read_file`. The JSON shape was more stable every time.

**My takeaway:**  
Best for this task. One example makes the format obvious and reduces invalid JSON risk.  
I would use few-shot whenever I need consistent output format.

---
## Experiment 3 — Chain-of-Thought (CoT) Prompting

**What is Chain-of-Thought?**  
We tell the model to "think step by step" before answering.  
This is useful for complex or ambiguous requests where the model needs to reason through multiple possibilities.

How it looks in the prompt:
```
Think briefly about intent, then return only final JSON.

User: Read file python_info.txt
```

In [ ]:
agent = AutomationAgent.create()

result_cot = agent.run(
    user_request="Read file python_info.txt",
    mode="once",
    prompt_style="cot",
)

print("=== CHAIN-OF-THOUGHT RESULT ===")
print(agent.result_as_json(result_cot))

### Chain-of-Thought Observations

- **Tool selected:** `read_file` ✅  
- **What I noticed:**
  - User message had the short "think then output JSON" line before the request.
  - The system prompt says "JSON only" so CoT reasoning stays internal to the model.
  - For a simple task like "read this file", CoT did not feel different from zero-shot in the final result.

**My takeaway:**  
CoT is best for **ambiguous or multi-step requests**, not simple single-tool tasks.  
Example: "Read the file, then summarise it, then email the summary" — here CoT would help the model plan all 3 steps.

---
## Side-by-Side Comparison

In [ ]:
styles = {
    "zero-shot": result_zero_shot,
    "few-shot": result_few_shot,
    "cot": result_cot,
}

print(f"{'Style':<12} {'Tool Selected':<20} {'Tool Success'}")
print("-" * 45)
for style, result in styles.items():
    tool = result.get("selected_tool", "?")
    success = result.get("tool_result", {}).get("success", "?")
    print(f"{style:<12} {tool:<20} {success}")

---
## Final Conclusion

For the request `"Read file python_info.txt"` across three runs:

| Prompt Style | Best For | Risk |
|---|---|---|
| **Zero-shot** | Simple, obvious tasks | Output format may vary |
| **Few-shot** | Consistent JSON format | Needs good examples |
| **Chain-of-Thought** | Complex / ambiguous tasks | Overkill for simple tasks |

**Winner for this task:** Few-shot — the example in the prompt matched our tool JSON and felt safest.

**When to use each (rule of thumb):**
- Zero-shot → quick tests, obvious single-word requests
- Few-shot → any time you need the LLM to stick to a strict output format
- CoT → multi-step reasoning, ambiguous inputs, complex decisions

---
## Bonus — Error Handling Demo

Week 1 deliverable: show graceful recovery from an API failure.

We ask the agent to read a file that does NOT exist — it should return `success: false` without crashing.

In [ ]:
agent = AutomationAgent.create()

# This file does not exist — the tool should return success: False gracefully
result_error = agent.run(
    user_request="Read file this_file_does_not_exist.txt",
    mode="once",
    prompt_style="zero-shot",
)

tool_result = result_error.get("tool_result", {})
print("Success:", tool_result.get("success"))   # Should be False
print("Message:", tool_result.get("message"))   # Should say 'File not found'
print("\nFull result:")
print(agent.result_as_json(result_error))